In [ ]:
%%capture 
# Import necessary libraries
import os, mne, re, csv, numpy as np, matplotlib.pyplot as plt
from glob import glob
from mne.channels import make_standard_montage
from mne.time_frequency import tfr_morlet
from eeg_function import drop_trials, single_trial_normalisation
# Set the derivatives directory path
root_derivatives = 'C:/Users/mfbpe/Desktop/DATA/2025_Valuation/derivatives/'

# Change the current working directory to the derivatives directory
os.chdir(root_derivatives)

mask_frontal = np.load("mask_frontal.npy")
mask_parietal = np.load("mask_parietal.npy")

all_files_path = sorted(glob(f"*raw.fif"), key=len)

datafile=open("data_TFR.csv","w", newline="")
writer=csv.writer(datafile, delimiter=";")
writer.writerow(["Participant", "reward", "difficulty", "Trial_TFR", "Outlier", "mean_FMtheta_frontal", "mean_FMtheta_parietal"])

event_id_stim = { 
    'stim/motiv/extr':11,'stim/motiv/hard':12,'stim/motiv/easy':13,'stim/amotiv/extr':14,'stim/amotiv/hard':15,'stim/amotiv/easy':16
}


for i,part in enumerate(all_files_path[:20]): #only first 20 participants for the sake of practice

    raw = mne.io.read_raw_fif(part, preload=True)
    
    events = mne.find_events(raw, shortest_event=1)            

    epochs= mne.Epochs(raw, events,baseline = (-.3, -.1), event_id=event_id_stim, picks='eeg',
            tmin=-1.6, tmax=2,preload=True, detrend=None, on_missing='ignore')

    if i==0:  
        sphere = mne.make_sphere_model('auto', 'auto', epochs.info)
        src = mne.setup_volume_source_space(sphere=sphere)
        forward = mne.make_forward_solution(epochs.info, trans=None, src=src, bem=sphere)

    epochs.set_eeg_reference('REST', forward=forward)
    

    chs_index = [i for i,x in enumerate(epochs.info['ch_names']) if x in ['FCz','Pz']]
            # Combine channels into one average

    epochs_comb = mne.channels.combine_channels(epochs, dict(Avg=chs_index))
    _,list_ti,_,_,_ = drop_trials(epochs_comb, do_mean=1, do_peak=1, do_slope=1, T1=-.3,T2=.5, chs='all')  
    
    epochs.drop(list_ti==0)

    # Time-frequency analysis (Morlet wavelet transform)
    freqs = np.logspace(np.log10(2), np.log10(30), num=80)
    n_cycles = np.logspace(np.log10(4), np.log10(14), 80)
    power = epochs.compute_tfr('morlet', freqs=freqs, n_cycles=n_cycles, use_fft=True, average=False, return_itc=False, decim=10, n_jobs=-1)

    power = single_trial_normalisation(power, tmin_baseline = -1.5, tmax_baseline = -.1, tmin_power = -1.5, tmax_power = 1.5) #use the method of Grandchamp & Delorme, baseline is classical baseline, power is the period of interest from baseline to a bit more than the expected signal

    power.crop(-.3,1.5)

    
    list_trigger_condition_level1 = [list(event_id_stim.keys())[list(event_id_stim.values()).index(x)].split("/")[1] for x in epochs.events[:,2]]
    list_trigger_condition_level2 = [list(event_id_stim.keys())[list(event_id_stim.values()).index(x)].split("/")[2] for x in epochs.events[:,2]]


    power_frontal = power.copy().pick(['FCz','Fz'])
    power_frontal.data = np.mean(power_frontal.data, axis=1)

    power_parietal = power.copy().pick(['CPz','Pz'])
    power_parietal.data = np.mean(power_parietal.data, axis=1)
    for ite_trial in range(len(epochs)):
        power_frontal_trial = np.mean(power_frontal.data[ite_trial][mask_frontal])
        power_parietal_trial = np.mean(power_parietal.data[ite_trial][mask_parietal])

        writer.writerow([part,  list_trigger_condition_level1[ite_trial], list_trigger_condition_level2[ite_trial],ite_trial, list_ti[ite_trial], round(power_frontal_trial,3), round(power_parietal_trial,3)])

    datafile.flush()
datafile.close()


In [21]:
np.mean(power_parietal.data[ite_trial][mask_parietal])

np.float64(-4.9921545941002385)

In [16]:
power_parietal_trial

np.float64(-5.151714164206997)